In [41]:
import pandas as pd
from nba_api.live.nba.endpoints.playbyplay import PlayByPlay

gameId = "0022200001"

pbp = PlayByPlay(game_id=gameId)
df = pd.DataFrame(pbp.actions.get_dict())

df.columns

Index(['actionNumber', 'clock', 'timeActual', 'period', 'periodType',
       'actionType', 'subType', 'qualifiers', 'personId', 'x', 'y',
       'possession', 'scoreHome', 'scoreAway', 'edited', 'orderNumber',
       'xLegacy', 'yLegacy', 'isFieldGoal', 'side', 'description',
       'personIdsFilter', 'teamId', 'teamTricode', 'descriptor',
       'jumpBallRecoveredName', 'jumpBallRecoverdPersonId', 'playerName',
       'playerNameI', 'jumpBallWonPlayerName', 'jumpBallWonPersonId',
       'jumpBallLostPlayerName', 'jumpBallLostPersonId', 'area', 'areaDetail',
       'shotDistance', 'shotResult', 'blockPlayerName', 'blockPersonId',
       'shotActionNumber', 'reboundTotal', 'reboundDefensiveTotal',
       'reboundOffensiveTotal', 'officialId', 'turnoverTotal', 'pointsTotal',
       'assistPlayerNameInitial', 'assistPersonId', 'assistTotal',
       'foulPersonalTotal', 'foulTechnicalTotal', 'foulDrawnPlayerName',
       'foulDrawnPersonId', 'stealPlayerName', 'stealPersonId'],
      dtype

In [2]:
from pathlib import Path
import requests


def clockToSeconds(clockValue):
    if pd.isna(clockValue):
        return pd.NA

    clockText = str(clockValue).replace('PT', '').replace('S', '')
    minutesText, secondsText = clockText.split('M', 1)
    return int(minutesText) * 60 + float(secondsText)


def secondsLeftInGame(periodValue, clockValue, periodTypeValue='REGULAR'):
    clockSeconds = clockToSeconds(clockValue)

    if str(periodTypeValue).upper() == 'OVERTIME' or periodValue > 4:
        overtimeNumber = max(periodValue - 5, 0)
        return overtimeNumber * 300 + clockSeconds

    return max(4 - periodValue, 0) * 720 + clockSeconds


def loadGameMeta(gameIdValue):
    candidateDirs = [Path('data/nba_gamelog'), Path('nba_gamelog')]
    gameIdValue = str(gameIdValue)

    gamelogPaths = []
    for candidateDir in candidateDirs:
        gamelogPaths.extend(sorted(candidateDir.glob('gamelog_*.parquet'), reverse=True))

    for gamelogPath in gamelogPaths:
        gamelogDf = pd.read_parquet(gamelogPath)
        gamelogDf.columns = gamelogDf.columns.str.lower().str.replace(r'_(\w)', lambda m: m.group(1).upper(), regex=True)
        gamelogDf['gameId'] = gamelogDf['gameId'].astype(str)

        gameRows = gamelogDf.loc[gamelogDf['gameId'] == gameIdValue, ['season', 'gameId', 'gameDate', 'matchup', 'teamId', 'teamAbbreviation', 'wl']].copy()
        if gameRows.empty:
            continue

        homeRow = gameRows.loc[gameRows['matchup'].str.contains(' vs. ', na=False)].head(1)
        awayRow = gameRows.loc[gameRows['matchup'].str.contains(' @ ', na=False)].head(1)

        if homeRow.empty:
            continue

        homeRow = homeRow.iloc[0]
        awayAbbreviation = awayRow.iloc[0]['teamAbbreviation'] if not awayRow.empty else homeRow['matchup'].split(' vs. ')[1]
        awayTeamId = awayRow.iloc[0]['teamId'] if not awayRow.empty else pd.NA

        return {
            'season': homeRow['season'],
            'gameId': homeRow['gameId'],
            'gameDate': pd.to_datetime(homeRow['gameDate']).date(),
            'matchup': homeRow['matchup'],
            'homeTeamId': homeRow['teamId'],
            'homeAbbreviation': homeRow['teamAbbreviation'],
            'awayTeamId': awayTeamId,
            'awayAbbreviation': awayAbbreviation,
            'homeWin': {'W': 1, 'L': 0}.get(homeRow['wl'], pd.NA)
        }

    searchedDirs = ', '.join(str(path) for path in candidateDirs)
    raise FileNotFoundError(f'No local gamelog entry found for gameId {gameIdValue} in {searchedDirs}')


df = df.copy()
df['gameId'] = str(gameId)
df['actionId'] = df['actionNumber']

for scoreColumn in ['scoreHome', 'scoreAway']:
    df[scoreColumn] = pd.to_numeric(df[scoreColumn], errors='coerce').ffill()

df['pointsTotal'] = df['scoreHome'] + df['scoreAway']
df['quarter'] = df['period']
df['secondsLeft'] = df.apply(
    lambda row: secondsLeftInGame(row['period'], row['clock'], row.get('periodType', 'REGULAR')),
    axis=1
)

gameMeta = loadGameMeta(gameId)
for columnName, columnValue in gameMeta.items():
    df[columnName] = columnValue

df['scoreDif'] = df['scoreHome'] - df['scoreAway']

if 'teamId' in df.columns:
    df['teamId'] = pd.to_numeric(df['teamId'], errors='coerce').astype('Int64')
    df['homeTeamId'] = pd.to_numeric(df['homeTeamId'], errors='coerce').astype('Int64')
    df['awayTeamId'] = pd.to_numeric(df['awayTeamId'], errors='coerce').astype('Int64')
    df['isHomeAction'] = df['teamId'].eq(df['homeTeamId'])
    df.loc[df['teamId'].isna(), 'isHomeAction'] = pd.NA
    df['actionTeamSide'] = pd.Series(pd.NA, index=df.index, dtype='object')
    df.loc[df['teamId'].eq(df['homeTeamId']), 'actionTeamSide'] = 'home'
    df.loc[df['teamId'].eq(df['awayTeamId']), 'actionTeamSide'] = 'away'

if 'possession' in df.columns:
    df['possession'] = pd.to_numeric(df['possession'], errors='coerce').astype('Int64')
    df['isHomePossession'] = df['possession'].eq(df['homeTeamId'])
    df.loc[df['possession'].isna(), 'isHomePossession'] = pd.NA
    df['possessionTeamSide'] = pd.Series(pd.NA, index=df.index, dtype='object')
    df.loc[df['possession'].eq(df['homeTeamId']), 'possessionTeamSide'] = 'home'
    df.loc[df['possession'].eq(df['awayTeamId']), 'possessionTeamSide'] = 'away'

preferredColumns = [
    'season',
    'gameId',
    'gameDate',
    'matchup',
    'homeTeamId',
    'homeAbbreviation',
    'awayTeamId',
    'awayAbbreviation',
    'homeWin',
    'actionId',
    'description',
    'quarter',
    'secondsLeft',
    'scoreHome',
    'scoreAway',
    'scoreDif',
    'pointsTotal',
    'teamId',
    'teamTricode',
    'actionTeamSide',
    'isHomeAction',
    'possession',
    'possessionTeamSide',
    'isHomePossession'
]

existingPreferredColumns = [column for column in preferredColumns if column in df.columns]
remainingColumns = [column for column in df.columns if column not in existingPreferredColumns]
df = df[existingPreferredColumns + remainingColumns].copy()

rotowireUrl = 'https://www.rotowire.com/betting/nba/tables/games-archive.php'
try:
    rotowireResponse = requests.get(rotowireUrl, timeout=30)
    rotowireResponse.raise_for_status()
    rotowireDf = pd.DataFrame(rotowireResponse.json())
    rotowireDf = rotowireDf[['game_date', 'home_team_abbrev', 'visit_team_abbrev', 'line']].rename(columns={
        'game_date': 'gameDate',
        'home_team_abbrev': 'homeAbbreviation',
        'visit_team_abbrev': 'awayAbbreviation'
    })
    rotowireDf['gameDate'] = pd.to_datetime(rotowireDf['gameDate']).dt.date
    rotowireDf = rotowireDf.drop_duplicates(subset=['gameDate', 'homeAbbreviation', 'awayAbbreviation'])
    df = df.merge(rotowireDf, on=['gameDate', 'homeAbbreviation', 'awayAbbreviation'], how='left')
except Exception as exc:
    print(f'Rotowire merge skipped: {exc}')

df.head()


,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,actionId,...,assistPlayerNameInitial,assistPersonId,assistTotal,foulPersonalTotal,foulTechnicalTotal,foulDrawnPlayerName,foulDrawnPersonId,stealPlayerName,stealPersonId,line
0,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.0
1,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.0
2,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.0
3,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.0
4,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.0


In [3]:
df.columns

Index(['season', 'gameId', 'gameDate', 'matchup', 'homeTeamId',
       'homeAbbreviation', 'awayTeamId', 'awayAbbreviation', 'homeWin',
       'actionId', 'description', 'quarter', 'secondsLeft', 'scoreHome',
       'scoreAway', 'scoreDif', 'pointsTotal', 'teamId', 'teamTricode',
       'actionTeamSide', 'isHomeAction', 'possession', 'possessionTeamSide',
       'isHomePossession', 'actionNumber', 'clock', 'timeActual', 'period',
       'periodType', 'actionType', 'subType', 'qualifiers', 'personId', 'x',
       'y', 'edited', 'orderNumber', 'xLegacy', 'yLegacy', 'isFieldGoal',
       'side', 'personIdsFilter', 'descriptor', 'jumpBallRecoveredName',
       'jumpBallRecoverdPersonId', 'playerName', 'playerNameI',
       'jumpBallWonPlayerName', 'jumpBallWonPersonId',
       'jumpBallLostPlayerName', 'jumpBallLostPersonId', 'area', 'areaDetail',
       'shotDistance', 'shotResult', 'blockPlayerName', 'blockPersonId',
       'shotActionNumber', 'reboundTotal', 'reboundDefensiveTotal',
   

In [4]:
cols_to_drop = ['isHomePossession', 'teamTricode', 'teamId', 'isHomeAction', 'clock', 'timeActual', 
                'periodType', 'qualifiers', 'x', 'y', 'edited', 'orderNumber', 'xLegacy', 'yLegacy', 
                'isFieldGoal', 'side', 'personIdsFilter', 'descriptor', 'jumpBallRecoveredName',
       'jumpBallRecoverdPersonId', 'playerNameI',
       'jumpBallWonPlayerName', 'jumpBallWonPersonId',
       'jumpBallLostPlayerName', 'jumpBallLostPersonId', 'area', 'areaDetail',
       'shotDistance','blockPlayerName', 'blockPersonId',
       'shotActionNumber', 'reboundTotal', 'reboundDefensiveTotal',
       'reboundOffensiveTotal', 'officialId', 'turnoverTotal',
       'assistPlayerNameInitial', 'assistPersonId', 'assistTotal','foulDrawnPlayerName',
       'foulDrawnPersonId', 'stealPlayerName', 'stealPersonId', 'actionId', 'actionNumber',
       'period']
df = df.drop(columns=cols_to_drop)

In [5]:
df.loc[df['actionType'] == 'foul'].head()

,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,...,possession,possessionTeamSide,actionType,subType,personId,playerName,shotResult,foulPersonalTotal,foulTechnicalTotal,line
13,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,J. Embiid shooting personal FOUL (1 PF) (Tatum...,...,1610612738,home,foul,personal,203954,Embiid,NaN,1.0,0.0,-3.0
24,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,P. Tucker personal FOUL (1 PF),...,1610612738,home,foul,personal,200782,Tucker,NaN,1.0,0.0,-3.0
30,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,T. Maxey charge offensive FOUL (1 PF),...,1610612755,away,foul,offensive,1630178,Maxey,NaN,1.0,0.0,-3.0
39,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,A. Horford shooting personal FOUL (1 PF) (Embi...,...,1610612755,away,foul,personal,201143,Horford,NaN,1.0,0.0,-3.0
45,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,A. Horford flagrant-type-1 personal FOUL (2 PF...,...,1610612755,away,foul,personal,201143,Horford,NaN,2.0,0.0,-3.0


In [11]:
df.columns

Index(['season', 'gameId', 'gameDate', 'matchup', 'homeTeamId',
       'homeAbbreviation', 'awayTeamId', 'awayAbbreviation', 'homeWin',
       'description', 'quarter', 'secondsLeft', 'scoreHome', 'scoreAway',
       'scoreDif', 'pointsTotal', 'actionTeamSide', 'possession',
       'possessionTeamSide', 'actionType', 'subType', 'personId', 'playerName',
       'shotResult', 'foulPersonalTotal', 'foulTechnicalTotal', 'line'],
      dtype='object')

In [14]:
foul_df = df.loc[df['actionType'] == 'foul', ['gameId', 'personId', 'actionTeamSide', 'playerName', 'quarter', 'secondsLeft', 'foulPersonalTotal']].copy()
foul_df = foul_df.rename(columns={'actionTeamSide' : 'playerHomeAway'})
foul_df.head()

,gameId,personId,playerHomeAway,playerName,quarter,secondsLeft,foulPersonalTotal
13,0022200001,203954,away,Embiid,1,2785.0,1.0
24,0022200001,200782,away,Tucker,1,2759.0,1.0
30,0022200001,1630178,away,Maxey,1,2737.0,1.0
39,0022200001,201143,home,Horford,1,2689.0,1.0
45,0022200001,201143,home,Horford,1,2664.0,2.0


Do we want to start counting bonus at 4 or 5 fouls? The 5th foul is the one they shoot the free throw on, but technically at 4 fouls, the next foul causes free throws. Right now, it is just counting at 5 fouls, but I may wish to change that

Need to account for bonus in overtime. Right now, it is not accounted for

In [17]:
# Build per-side foul event lists from foul_df
home_foul_events = foul_df[foul_df['playerHomeAway'] == 'home'][['quarter', 'secondsLeft']].copy()
away_foul_events = foul_df[foul_df['playerHomeAway'] == 'away'][['quarter', 'secondsLeft']].copy()

# For each action in df, count how many fouls each side has committed in that quarter
# up to (and including) that moment.  Since secondsLeft decreases as the game progresses,
# a foul happened at-or-before the current action when its secondsLeft >= current secondsLeft.
df_idx = df[['quarter', 'secondsLeft']].reset_index()  # preserves original df index

home_merged = df_idx.merge(home_foul_events, on='quarter', suffixes=('', '_foul'))
home_merged = home_merged[home_merged['secondsLeft_foul'] >= home_merged['secondsLeft']]
df['homeFouls'] = home_merged.groupby('index').size().reindex(df.index, fill_value=0)

away_merged = df_idx.merge(away_foul_events, on='quarter', suffixes=('', '_foul'))
away_merged = away_merged[away_merged['secondsLeft_foul'] >= away_merged['secondsLeft']]
df['awayFouls'] = away_merged.groupby('index').size().reindex(df.index, fill_value=0)

# homeBonus: away team has 5+ fouls in the quarter → home team shoots free throws
# awayBonus: home team has 5+ fouls in the quarter → away team shoots free throws
df['homeBonus'] = (df['awayFouls'] >= 5).astype(int)
df['awayBonus'] = (df['homeFouls'] >= 5).astype(int)

df.head(10)


,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,...,personId,playerName,shotResult,foulPersonalTotal,foulTechnicalTotal,line,homeFouls,awayFouls,homeBonus,awayBonus
0,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,Period Start,...,0,NaN,NaN,NaN,NaN,-3.0,0,0,0,0
1,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,Jump Ball J. Embiid vs. A. Horford: Tip to T. ...,...,202699,Harris,NaN,NaN,NaN,-3.0,0,0,0,0
2,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,MISS J. Embiid 12' turnaround fadeaway Shot - ...,...,203954,Embiid,Missed,NaN,NaN,-3.0,0,0,0,0
3,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,J. Brown BLOCK (1 BLK),...,1627759,Brown,NaN,NaN,NaN,-3.0,0,0,0,0
4,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,P. Tucker REBOUND (Off:1 Def:0),...,200782,Tucker,NaN,NaN,NaN,-3.0,0,0,0,0
5,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,P. Tucker lost ball out-of-bounds TURNOVER (1 TO),...,200782,Tucker,NaN,NaN,NaN,-3.0,0,0,0,0
6,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,M. Smart 13' driving floating bank Jump Shot (...,...,203935,Smart,Made,NaN,NaN,-3.0,0,0,0,0
7,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,MISS T. Harris 14' driving floating Shot,...,202699,Harris,Missed,NaN,NaN,-3.0,0,0,0,0
8,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,T. Harris REBOUND (Off:1 Def:0),...,202699,Harris,NaN,NaN,NaN,-3.0,0,0,0,0
9,2022-23,0022200001,2022-10-18,BOS vs. PHI,1610612738,BOS,1610612755,PHI,1,T. Harris tip Layup (2 PTS),...,202699,Harris,Made,NaN,NaN,-3.0,0,0,0,0


In [33]:
foul_descriptions = df.loc[df['actionType'] == 'foul', 'description']
foul_descriptions

13     J. Embiid shooting personal FOUL (1 PF) (Tatum...
24                        P. Tucker personal FOUL (1 PF)
30                 T. Maxey charge offensive FOUL (1 PF)
39     A. Horford shooting personal FOUL (1 PF) (Embi...
45     A. Horford flagrant-type-1 personal FOUL (2 PF...
51                        N. Vonleh personal FOUL (1 PF)
60     J. Embiid shooting personal FOUL (2 PF) (Smart...
66             N. Vonleh loose ball personal FOUL (2 PF)
73     J. Brown transition take personal FOUL (1 PF) ...
97         M. Brogdon personal FOUL (1 PF) (Harris 2 FT)
115    G. Williams shooting personal FOUL (1 PF) (Har...
126    M. Harrell shooting personal FOUL (1 PF) (Grif...
137    B. Griffin shooting personal FOUL (1 PF) (Hard...
146    D. White shooting personal FOUL (1 PF) (House ...
152    D. Melton shooting personal FOUL (1 PF) (Tatum...
171                     G. Williams personal FOUL (2 PF)
177                        T. Maxey personal FOUL (2 PF)
179    G. Williams shooting per

Make sure techs aren't counted as personal fouls

In [38]:
# Can take out .head() to show all actions and subtypes
df[['actionType', 'subType']].drop_duplicates().reset_index(drop=True)

,actionType,subType
0,period,start
1,jumpball,recovered
2,2pt,Jump Shot
3,block,
4,rebound,offensive
5,turnover,out-of-bounds
6,2pt,Layup
7,3pt,Jump Shot
8,rebound,defensive
9,foul,personal


In [37]:
import re

# Matches "(M FT)" in a foul description, e.g. "(Tatum 2 FT)"
ft_awarded_pattern = re.compile(r'\((\w[\w\s\.]*\s)?(\d+)\s+FT\)')
# Matches "X of M" in a freethrow subType, e.g. "1 of 2"
ft_attempt_pattern = re.compile(r'(\d+)\s+of\s+(\d+)')

home_ft, away_ft = 0, 0
home_ft_list, away_ft_list = [], []

for _, row in df.iterrows():
    action = str(row['actionType']) if pd.notna(row['actionType']) else ''
    sub    = str(row['subType'])    if pd.notna(row['subType'])    else ''
    desc   = str(row['description']) if pd.notna(row['description']) else ''
    side   = row['actionTeamSide']  # 'home', 'away', or NaN — side of the acting player

    # Sanity check: both counters must be 0 at the start of a new period
    if action == 'period' and sub == 'start':
        if home_ft != 0 or away_ft != 0:
            raise ValueError(
                f"Period started with unresolved free throws: "
                f"homeFreeThrows={home_ft}, awayFreeThrows={away_ft}"
            )

    # Technical foul: always awards 1 FT to the opposing team
    elif action == 'foul' and sub == 'technical':
        if side == 'home':
            away_ft = 1
        elif side == 'away':
            home_ft = 1

    # Non-technical foul: parse "(M FT)" from description
    elif action == 'foul':
        ft_match = ft_awarded_pattern.search(desc)
        if ft_match:
            m = int(ft_match.group(2))   # number of FTs awarded
            if side == 'home':
                away_ft = m   # home player fouled → away team shoots
            elif side == 'away':
                home_ft = m   # away player fouled → home team shoots

    # Free throw attempt: decrement the shooting team's remaining count
    elif action == 'freethrow':
        of_match = ft_attempt_pattern.search(sub)
        if of_match:
            x = int(of_match.group(1))   # this attempt number
            m = int(of_match.group(2))   # total attempts
            remaining = m - x
            if side == 'home':
                home_ft = remaining
            elif side == 'away':
                away_ft = remaining

    home_ft_list.append(home_ft)
    away_ft_list.append(away_ft)

df['homeFreeThrows'] = home_ft_list
df['awayFreeThrows'] = away_ft_list

# Verify: show foul rows and the immediately following free throw rows
sample_idx = df.index[df['actionType'].isin(['foul', 'freethrow'])].tolist()[:30]
df.loc[sample_idx, ['description', 'actionType', 'subType', 'actionTeamSide', 'homeFreeThrows', 'awayFreeThrows']]


,description,actionType,subType,actionTeamSide,homeFreeThrows,awayFreeThrows
13,J. Embiid shooting personal FOUL (1 PF) (Tatum...,foul,personal,away,2,0
14,J. Tatum Free Throw 1 of 2 (4 PTS),freethrow,1 of 2,home,1,0
15,J. Tatum Free Throw 2 of 2 (5 PTS),freethrow,2 of 2,home,0,0
24,P. Tucker personal FOUL (1 PF),foul,personal,away,0,0
30,T. Maxey charge offensive FOUL (1 PF),foul,offensive,away,0,0
39,A. Horford shooting personal FOUL (1 PF) (Embi...,foul,personal,home,0,2
40,MISS J. Embiid Free Throw 1 of 2,freethrow,1 of 2,away,0,1
42,J. Embiid Free Throw 2 of 2 (1 PTS),freethrow,2 of 2,away,0,0
45,A. Horford flagrant-type-1 personal FOUL (2 PF...,foul,personal,home,0,3
46,J. Harden flagrant Free Throw 1 of 3 (1 PTS),freethrow,1 of 3,away,0,2


## May need to account for flagrant foul rules

### Need to add:
- possession (need to handle case where neither team has outright possession, like for missed shots)
- free throws (added, need to account for flagrant fouls)
- ejections